In [1]:
import os
import json
import math
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from scipy.stats import t

NOME_BASE = 'wikidata'
SPLITS = [1, 2, 3, 4, 5]

"""
RELATION_STATS = {
    'conjugue': {'mean': -4.42, 'std': 10.45},
    'filho': {'mean': -35.26, 'std': 9.3},
    'irmao': {'mean': -0.48, 'std': 9.64},
    'mae': {'mean': 28.55, 'std': 7.6},
    'pai': {'mean': 34.54, 'std': 9.1},
    'parente': {'mean': -3.05, 'std': 49.08}
}
"""

EMB_DIRS = {
    'profissao': ('occupation_embeddings_br', 'Occupation: ', 'emb_Profissoes_base'),
    'endereço': ('residence_embeddings_br', 'Residence: ', 'emb_Enderecos_base'),
    'local_nascimento': ('birthplace_embeddings_br', 'Birthplace: ', 'emb_Locais_Nascimento_base'),
    'nacionalidade': ('nationality_embeddings_br', 'Nationality: ', 'emb_Nacionalidades_base')
}

print("Bibliotecas importadas.")

Bibliotecas importadas.


In [2]:
embeddings_cache = {}

for campo, (folder, prefixo, col_name) in EMB_DIRS.items():
    embeddings_cache[campo] = {}
    if os.path.exists(folder):
        dataset = pq.ParquetDataset(folder)
        df_emb = dataset.read().to_pandas()
        col_text = [c for c in df_emb.columns if not c.startswith('emb_')][0]
        
        for _, row in df_emb.iterrows():
            embeddings_cache[campo][row[col_text]] = np.array(row[col_name], dtype=np.float32)
            
print("Embeddings carregados para a RAM.")

Embeddings carregados para a RAM.


In [3]:
def _get_year(data_str):
    try:
        if pd.isna(data_str) or str(data_str).lower() == 'nan': return None
        return int(str(data_str).strip()[:4])
    except: return None

# compara duas listas (ex: profissões de A com as profissões de B) e retorna a maior similaridade encontrada entre qualquer par
def calc_max_cosseno_vetorizado(lista1, lista2, campo):
    if not lista1 or not lista2: return 0.0
    prefix = EMB_DIRS[campo][1]
    
    embs1 = [embeddings_cache[campo].get(f"{prefix}{i}") for i in lista1]
    embs2 = [embeddings_cache[campo].get(f"{prefix}{i}") for i in lista2]
    
    embs1 = [e for e in embs1 if e is not None]
    embs2 = [e for e in embs2 if e is not None]
    
    if not embs1 or not embs2: return 0.0
    
    # 1. Converte para array e substitui qualquer NaN ou Inf que já venha do parquet por 0.0
    A = np.nan_to_num(np.array(embs1), nan=0.0, posinf=0.0, neginf=0.0)
    B = np.nan_to_num(np.array(embs2), nan=0.0, posinf=0.0, neginf=0.0)
    
    norm_A = np.linalg.norm(A, axis=1, keepdims=True)
    norm_B = np.linalg.norm(B, axis=1, keepdims=True)
    
    # 2. Usa np.clip para garantir que NENHUM valor seja menor que 1e-10
    # Isso é mais seguro que 'norm == 0', pois protege contra floats minúsculos (ex: 1e-300)
    norm_A = np.clip(norm_A, a_min=1e-10, a_max=None)
    norm_B = np.clip(norm_B, a_min=1e-10, a_max=None)
    
    # 3. Calcula o cosseno
    cossenos = np.dot(A / norm_A, (B / norm_B).T)
    
    # 4. Última barreira: limpa o resultado do produto escalar limitando entre -1 e 1 (regras do cosseno)
    cossenos = np.nan_to_num(cossenos, nan=0.0)
    cossenos = np.clip(cossenos, a_min=-1.0, a_max=1.0)
    
    return float(np.max(cossenos))

def pontuar_heuristica(pessoa_contexto, candidato, parentesco):
    # age score com t-student
    y_ctx = _get_year(pessoa_contexto.get('data_nascimento'))
    y_cand = _get_year(candidato.get('data_nascimento'))
    diff_idade = (y_cand - y_ctx) if (y_ctx and y_cand) else None
    
    stats = RELATION_STATS.get(parentesco.lower())
    
    if diff_idade is not None:
        graus_liberdade = stats['df']
        localizacao = stats['loc']
        escala = stats['scale']
        
        altura = t.pdf(diff_idade, df=graus_liberdade, loc=localizacao, scale=escala)
        altura_max = t.pdf(localizacao, df=graus_liberdade, loc=localizacao, scale=escala)
        
        age_score = float(altura / altura_max) if altura_max > 0 else 0.0
    else:
        age_score = 0.0

    prof_score = calc_max_cosseno_vetorizado(
        pessoa_contexto.get('profissao', []), 
        candidato.get('profissao', []), 
        'profissao'
    )
    
    # pega o maior cosseno entre endereço, nascimento e nacionalidade
    end_score = calc_max_cosseno_vetorizado(pessoa_contexto.get('endereço', []), candidato.get('endereço', []), 'endereço')
    nasc_score = calc_max_cosseno_vetorizado(pessoa_contexto.get('local_nascimento', []), candidato.get('local_nascimento', []), 'local_nascimento')
    nac_score = calc_max_cosseno_vetorizado(pessoa_contexto.get('nacionalidade', []), candidato.get('nacionalidade', []), 'nacionalidade')
    
    cosseno_endereco_max = max(end_score, nasc_score, nac_score)
    
    score_final = (0.50 * age_score) + (0.40 * cosseno_endereco_max) + (0.10 * prof_score)
    
    return score_final

print("Funções preparadas.")

Funções preparadas.


In [4]:
# inferência da heurística nessa célula roda apenas nos splits de teste, acho que poderia ser no arquivo original também
resultados_por_split = []

def calcular_mrr(posicao):
    return 1.0 / posicao if posicao > 0 else 0.0

for split in SPLITS:
    print(f"\nCalculando Heurística - Split {split}...")
    test_file = f"splits/{NOME_BASE}_split{split}_test.jsonl"
    split_results = []

    with open(f"splits/stats_split{split}.json", 'r') as f:
        RELATION_STATS = json.load(f)
    
    with open(test_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc=f"Split {split}"):
            reg = json.loads(line)
            
            pessoa_alvo = reg.get('pessoa_ground_truth', reg.get('Dados_pessoa_1')) # original
            pessoa_contexto = reg.get('candidato_correto', reg.get('Dados_pessoa_2')) # parente
            homonimos = reg.get('lista_homonimos', reg.get('Lista_homonimos'))
            parentesco = reg.get('parentesco', reg.get('Parentesco'))
            
            resultados_ranking = []
            
            for h in homonimos:
                is_correto = (str(h.get('id')) == str(pessoa_alvo.get('id')))
                score = pontuar_heuristica(pessoa_contexto, h, parentesco)
                resultados_ranking.append((h.get('id'), score, is_correto))
                
            resultados_ranking.sort(key=lambda x: x[1], reverse=True)
            
            posicao_correta = -1
            for idx, (_, _, is_p2) in enumerate(resultados_ranking):
                if is_p2:
                    posicao_correta = idx + 1
                    break
                    
            linha_resultado = {
                "id_pessoa_1": pessoa_alvo.get('id'),
                "Parentesco": parentesco,
                "MRR": calcular_mrr(posicao_correta)
            }
            
            for k in [1, 5, 10, 15, 20]:
                linha_resultado[f"Recall@{k}"] = 1.0 if (posicao_correta > 0 and posicao_correta <= k) else 0.0
                
            split_results.append(linha_resultado)
            
    df_metrics = pd.DataFrame(split_results)
    colunas_metricas = [col for col in df_metrics.columns if 'Recall' in col or 'MRR' in col]
    medias_split = df_metrics[colunas_metricas].mean().to_dict()
    medias_split['Split'] = split
    resultados_por_split.append(medias_split)

df_final = pd.DataFrame(resultados_por_split)

cols = ['Split', 'MRR', 'Recall@1', 'Recall@5', 'Recall@10', 'Recall@15', 'Recall@20']
df_final = df_final[cols]

media_geral = df_final.mean().to_dict()
media_geral['Split'] = 'Média'
df_final.loc[len(df_final)] = media_geral

print("HEURÍSTICA: Resultados Processamento das bases para o artigo SBBD 2026")
display(df_final.round(4))


Calculando Heurística - Split 1...


Split 1: 0it [00:00, ?it/s]


Calculando Heurística - Split 2...


Split 2: 0it [00:00, ?it/s]


Calculando Heurística - Split 3...


...

Split 4: 0it [00:00, ?it/s]


Calculando Heurística - Split 5...


Split 5: 0it [00:00, ?it/s]

HEURÍSTICA: Resultados Processamento das bases para o artigo SBBD 2026


,Split,MRR,Recall@1,Recall@5,Recall@10,Recall@15,Recall@20
0,1,0.8926,0.8248,0.9741,0.9872,0.9912,0.9933
1,2,0.8915,0.8218,0.9757,0.9881,0.9919,0.9938
2,3,0.8933,0.8248,0.9759,0.9880,0.9916,0.9935
3,4,0.8905,0.8209,0.9752,0.9875,0.9914,0.9934
4,5,0.8900,0.8200,0.9746,0.9871,0.9907,0.9928
5,Média,0.8916,0.8224,0.9751,0.9876,0.9913,0.9933
